# Build Custom Components

This notebook checks that the tutorial package registers one custom implementation for each component namespace covered by the manual.

The important pattern is the decorator in each source file, for example `@ARCHITECTURE_REGISTRY.register("sensor_health_mlp")`. Importing the package imports the grouped modules, and importing those modules runs the decorators.

> **Notebook memory:** Importing the scientific Python stack (for example PyTorch, NumPy, Matplotlib, and ZenML) can keep approximately 1 GiB of RAM assigned to this kernel for its lifetime. Python cannot safely unload native extension modules. **After finishing this tutorial, restart its kernel to clear imported libraries and release that RAM** (or shut down the kernel entirely). Restarting keeps the notebook open with a fresh, low-memory kernel. Do this before running several tutorial notebooks at once.

In [ ]:
from pathlib import Path
import sys

# Notebooks are often opened from inside the package directory. This small
# bootstrap lets the same notebook work from a source checkout before the
# package is installed in editable mode.
for root in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    for src in (root / "src", root / "plugins" / "example_plugin" / "src"):
        if (src / "pioneerml_example_plugin").exists() or (src / "pioneerml").exists():
            src_text = str(src)
            if src_text not in sys.path:
                sys.path.insert(0, src_text)

import pioneerml_example_plugin.components_tutorial_examples  # imports grouped modules and registers plugins
from pioneerml.plugin import resolve_plugin

expected = {
    "architecture": "sensor_health_mlp",
    "compiler": "sensor_health_noop",
    "evaluator": "sensor_health_evaluator",
    "exporter": "sensor_health_state_dict",
    "hpo": "sensor_health_hpo",
    "input_backend": "sensor_csv",
    "inference_batch_executor": "sensor_health_batch_executor",
    "loader": "sensor_health_loader",
    "loader_manager": "sensor_tutorial",
    "loss": "sensor_health_bce",
    "metric": "sensor_health_accuracy",
    "model_handle": "sensor_health_state_dict",
    "module": "sensor_health_module",
    "objective": "sensor_health_val_loss",
    "output_backend": "sensor_jsonl",
    "plot": "sensor_health_score_histogram",
    "search_parameter": "sensor_centered_float",
    "search_space": "sensor_health_search_space",
    "trainer": "sensor_health_trainer",
    "writer": "sensor_health_writer",
}

resolved = {namespace: resolve_plugin(namespace=namespace, name=name) for namespace, name in expected.items()}
resolved

## Reading The Source

Start with these grouped packages when learning the extension points:

- `data_loading/`: data comes in as Arrow tables, moves through loader stages, and becomes PyG graph batches.
- `modeling/`: the standard training step builds model, compiler, module, and trainer from config.
- `evaluation/`: the evaluator gathers tensors, then registered metric and plot plugins consume a shared context.
- `export/`: export writes a state dict plus architecture config, and the model handle reverses that process.
- `inference/`: the batch executor delegates the standard inference sequence and validates model-specific outputs.
- `tuning/`: HPO owns search suggestions and objective extraction.
- `writing/`: writer stages build an Arrow prediction table, and an output backend serializes it.

The next cell prints a loader stage because stages are one of the clearest places to see the input-output state contract.

In [ ]:
from inspect import getsource

from pioneerml_example_plugin.components_tutorial_examples.data_loading.loader import SensorGraphBuildStage

print(getsource(SensorGraphBuildStage))